# 02 — Feature Store

**Objetivo:** construir variáveis comportamentais **históricas** por cliente × snapshot, reutilizáveis e sem vazamento temporal.

Todas as features usam exclusivamente a janela $[t-5\text{d},\, t-1\text{d}]$ — **nunca** dados de $t$ em diante.

**Variáveis criadas:**

| Grupo | Variáveis |
| --- | --- |
| Recência | `recencia_dias` (dias desde a última compra) |
| Frequência | `freq_compras_5d`, `tendencia_2d_vs_3a5` |
| Monetário | `valor_total_5d`, `ticket_medio_5d` |
| Volume | `qtde_itens_5d`, `preco_medio_5d` |
| Diversidade | `produtos_distintos_5d`, `franquias_distintas_5d` |
| Perfil | `pagamento_predominante` (moda dos métodos de pagamento) |

**Output:** `workspace.ml_bakehouse.customer_features`

In [0]:
# ── Configuração central (idem notebook 00) ──
SOURCE  = "samples.bakehouse.sales_transactions"
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"
HIST_WINDOW = 5

from pyspark.sql import functions as F

population = spark.table(f"{CATALOG}.{SCHEMA}.population_active")
trans = (
    spark.table(SOURCE)
    .drop("cardNumber")
    .withColumn("tx_date", F.to_date("dateTime"))
)
print("População:", population.count(), "linhas | Transações:", trans.count())

In [0]:
# Features comportamentais na janela histórica [t-5, t-1]
features = (
    trans.alias("t")
    .join(
        population.alias("p"),
        on=(F.col("t.customerID") == F.col("p.customerID"))
         & (F.col("t.tx_date") >= F.col("p.reference_date") - F.lit(HIST_WINDOW))
         & (F.col("t.tx_date") <= F.col("p.reference_date") - F.lit(1)),
        how="inner",
    )
    .groupBy(F.col("p.reference_date"), F.col("t.customerID"))
    .agg(
        F.datediff(F.col("p.reference_date"), F.max("t.tx_date")).alias("recencia_dias"),
        F.countDistinct("t.transactionID").alias("freq_compras_5d"),
        F.sum("t.totalPrice").alias("valor_total_5d"),
        F.round(F.sum("t.totalPrice") / F.countDistinct("t.transactionID"), 2).alias("ticket_medio_5d"),
        F.sum("t.quantity").alias("qtde_itens_5d"),
        F.countDistinct("t.product").alias("produtos_distintos_5d"),
        F.countDistinct("t.franchiseID").alias("franquias_distintas_5d"),
        F.round(F.avg("t.unitPrice"), 2).alias("preco_medio_5d"),
        # Tendência: compras nos últimos 2 dias vs. dias 3–5 anteriores à referência
        F.sum(F.when(F.col("t.tx_date") >= F.col("p.reference_date") - F.lit(2), 1).otherwise(0)).alias("compras_ultimos_2d"),
        F.sum(F.when(F.col("t.tx_date") <  F.col("p.reference_date") - F.lit(2), 1).otherwise(0)).alias("compras_dias_3a5"),
        F.mode("t.paymentMethod").alias("pagamento_predominante"),
    )
)

# Índice de tendência: razão entre as frequências (suavizada)
features = features.withColumn(
    "tendencia_2d_vs_3a5",
    F.round(F.col("compras_ultimos_2d") / (F.col("compras_dias_3a5") + 1), 3),
).drop("compras_ultimos_2d", "compras_dias_3a5")

print("Features calculadas:", features.count(), "linhas")

In [0]:
# Verificação 1: cobertura — toda a população deve ter features (mesmo nº de linhas)
print("Linhas população:", population.count(), "| Linhas features:", features.count())
assert population.count() == features.count(), "Cobertura divergente da população!"

# Verificação 2: sanidade das features (describe)
display(features.drop("pagamento_predominante").summary())

# Verificação 3: recência nunca pode ser 0 ou negativa (leakage do dia da referência)
print("Recência mínima observada (esperado >= 1):",
      features.agg(F.min("recencia_dias")).collect()[0][0])

In [0]:
# Gravação como Delta Table (futuro candidato a Feature Table no Unity Catalog)
(features.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.customer_features"))

print(f"Salvo: {CATALOG}.{SCHEMA}.customer_features")
display(spark.table(f"{CATALOG}.{SCHEMA}.customer_features").orderBy("reference_date", "customerID").limit(10))

## Notas

* `F.mode()` calcula a moda de `paymentMethod` na janela — disponível no PySpark 3.4+ (DBR 14.x). Se der erro na sua versão, substitua por `F.max(F.struct(F.count(...)))`.
* Esta tabela pode ser registrada posteriormente como **Feature Table** no Unity Catalog para reuso por outros modelos (Feature Engineering client).
* **Próximo passo:** `03_target` — variável resposta com a janela futura de 1 dia.